In [1]:
import pandas as pd
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
import os
import sys
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data, write_to_hdf5_file

from plot_icusleep_data import icu_sleep_plot

from scipy.stats import variation

In [ ]:
cohort = 'icu'

data_dir = f'/media/ssd2/{cohort}_breathing'
output_dir = f'/media/ssd2/{cohort}_breathing_instability_index'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    

files = os.listdir(data_dir)
files.sort()

for file in files:

    try:
        if os.path.exists(os.path.join(output_dir, file)):
            continue

        data, hdr = load_sleep_data(os.path.join(data_dir, file), idx_to_datetime=1)
        study_id = cohort + '_' + str(hdr['study_id']).zfill(3)
        fs = hdr['fs']

        data = compute_breathing_instability_index_routine(data)

        hdr['study_id'] = np.int32( hdr['study_id'])
        hdr['MRN'] = np.int32(hdr['MRN'])
        hdr['fs'] = np.int32(hdr['fs'])
        hdr['start_datetime'] = np.array([hdr['start_datetime'].year,hdr['start_datetime'].month,hdr['start_datetime'].day, hdr['start_datetime'].hour, hdr['start_datetime'].minute, hdr['start_datetime'].second, hdr['start_datetime'].microsecond])

        write_to_hdf5_file(data, os.path.join(output_dir, file), hdr=hdr)  

    except Exception as e:
        print(file)
        print(e)
        continue


In [ ]:
files = os.listdir(data_dir)
file = files[0]
data, hdr = load_sleep_data(os.path.join(data_dir, file), idx_to_datetime=1)
study_id = cohort + '_' + str(hdr['study_id']).zfill(3)
fs = hdr['fs']

files = os.listdir(data_dir)
file = files[1]
data2, hdr2 = load_sleep_data(os.path.join(data_dir, file), idx_to_datetime=1)
study_id2 = cohort + '_' + str(hdr2['study_id']).zfill(3)

In [44]:
data.columns

Index(['IBI', 'IBI_mean_5min', 'IBI_std_5min', 'TVpB', 'TVpB_regularity',
       'TVpB_regularity_10sec', 'abd', 'acc_ai_10sec', 'acc_ai_1sec',
       'acc_enmo', 'acc_enmo_10sec', 'acc_enmo_1sec', 'accx', 'accx_1sec',
       'accy', 'accy_1sec', 'accz', 'accz_1sec', 'annotation', 'apnea', 'band',
       'cflow', 'chest', 'crcstatus', 'deriv1', 'epoch', 'exht', 'hypo_10_60',
       'inht', 'inht_cycle_ratio', 'inht_cycle_ratio_10sec', 'inht_exht_ratio',
       'inht_exht_ratio_10sec', 'katz_fd_10s_smoothed', 'katz_fd_30s_smoothed',
       'katz_fd_45s_smoothed', 'katz_fd_60s_smoothed', 'movavg_0_5s',
       'movavg_1_2s', 'movavg_1min', 'movmedian_10min', 'movmedian_1min',
       'movmedian_30sec', 'movmedian_5min', 'movstd_10min', 'movstd_10s',
       'movstd_12s', 'movstd_15s', 'movstd_1min', 'movstd_30s', 'movstd_30sec',
       'movstd_5min', 'movstd_60s', 'movstd_8s', 'peaks', 'position_cluster',
       'pr', 'ptaf', 'pulsequality', 'rr_10s', 'rr_10s_smooth', 'rr_60s',
       'samp

In [11]:
data.inht_cycle_ratio.dropna()

2019-05-03 22:57:41.700    0.166626
2019-05-03 22:57:51.600    0.339355
2019-05-03 22:57:56.900    0.302002
2019-05-03 22:58:02.100    0.254150
2019-05-03 22:58:07.900    0.291748
                             ...   
2019-05-04 06:17:22.400    0.077637
2019-05-04 06:17:45.800    0.450684
2019-05-04 06:17:50.400    0.233276
2019-05-04 06:17:54.100    0.089172
2019-05-04 06:18:18.800    0.697754
Name: inht_cycle_ratio, Length: 4233, dtype: float32

In [12]:
from plotly.offline import plot

In [13]:
data.columns

Index(['IBI', 'IBI_mean_5min', 'IBI_std_5min', 'TVpB', 'TVpB_regularity',
       'TVpB_regularity_10sec', 'abd', 'acc_ai_10sec', 'acc_ai_1sec',
       'acc_enmo', 'acc_enmo_10sec', 'acc_enmo_1sec', 'accx', 'accx_1sec',
       'accy', 'accy_1sec', 'accz', 'accz_1sec', 'annotation', 'apnea', 'band',
       'cflow', 'chest', 'crcstatus', 'deriv1', 'epoch', 'exht', 'hypo_10_60',
       'inht', 'inht_cycle_ratio', 'inht_cycle_ratio_10sec', 'inht_exht_ratio',
       'inht_exht_ratio_10sec', 'katz_fd_10s_smoothed', 'katz_fd_30s_smoothed',
       'katz_fd_45s_smoothed', 'katz_fd_60s_smoothed', 'movavg_0_5s',
       'movavg_1_2s', 'movavg_1min', 'movmedian_10min', 'movmedian_1min',
       'movmedian_30sec', 'movmedian_5min', 'movstd_10min', 'movstd_10s',
       'movstd_12s', 'movstd_15s', 'movstd_1min', 'movstd_30s', 'movstd_30sec',
       'movstd_5min', 'movstd_60s', 'movstd_8s', 'peaks', 'position_cluster',
       'pr', 'ptaf', 'pulsequality', 'rr_10s', 'rr_10s_smooth', 'rr_60s',
       'samp

In [69]:
signals_to_plot = ['acc_ai_10sec',
                   'movavg_0_5s', 
                   'movavg_1min',  
                   'rr_10s_smooth', 
                   'IBI', 
                   'IBI_mean_5min',
#                    'IBI_std_5min',
#                    'IBI_std_2min',
                    'IBI_cvar_30sec',
#                    'IBI_cvar_1min',
#                    'IBI_cvar_2min',
#                    'IBI_cvar_5min',
#                    'ventilation_cvar_30sec',
#                    'ventilation_cvar_1min',
                   'ventilation_cvar_2min',
#                    'ventilation_cvar_5min',
#                    'ventilation_std_2min',
                   'ventilation_10s_smooth',
                   'ventilation_2min',
                   'instability_index_30sec',
                   'instability_index_1min',
                   'instability_index_2min',
                   'instability_index_5min',
                   'inht_cycle_ratio_10sec']

In [ ]:
# data2['ventilation_2min'] = data2['ventilation0'].rolling('2min').sum()/2
# data2['ventilation_std_2min'] = data2['ventilation_2min'].reset_index()['ventilation_2min'].rolling(120*fs, center=True, min_periods=1).std().values


In [66]:
data['ventilation_10s']   = data['ventilation0'].reset_index(drop=True).rolling(10*fs, center=True, min_periods=1).sum().values*6
data['ventilation_10s_smooth'] = data['ventilation_10s'].reset_index(drop=True).rolling(6*fs, center=True, min_periods=1).mean().values

data2['ventilation_10s']   = data2['ventilation0'].reset_index(drop=True).rolling(10*fs, center=True, min_periods=1).sum().values*6
data2['ventilation_10s_smooth'] = data2['ventilation_10s'].reset_index(drop=True).rolling(6*fs, center=True, min_periods=1).mean().values


def compute_breathing_instability_index(data, vname='2min', sec=120):
    data[f'ventilation_cvar_{vname}'] = data[f'ventilation_10s_smooth'].reset_index(drop=True).rolling(sec*fs, center=True, min_periods=1).apply(lambda x: variation(x, nan_policy='omit'), raw=False).values
    data[f'IBI_cvar_{vname}'] = data['IBI'].reset_index()['IBI'].rolling(sec*fs, center=True, min_periods=1).apply(lambda x: variation(x, nan_policy='omit'), raw=False).values
    data[f'instability_index_{vname}'] = (data[f'ventilation_cvar_{vname}'].values + data[f'IBI_cvar_{vname}'].values)/2
    return data

vname='30sec'
sec=30
data = compute_breathing_instability_index(data, vname=vname, sec=sec)

vname='1min'
sec=60
data = compute_breathing_instability_index(data, vname=vname, sec=sec)

vname='2min'
sec=120
data = compute_breathing_instability_index(data, vname=vname, sec=sec)

vname='5min'
sec=300
data = compute_breathing_instability_index(data, vname=vname, sec=sec)

vname='30sec'
sec=30
data2 = compute_breathing_instability_index(data2, vname=vname, sec=sec)

vname='1min'
sec=60
data2 = compute_breathing_instability_index(data2, vname=vname, sec=sec)

vname='2min'
sec=120
data2 = compute_breathing_instability_index(data2, vname=vname, sec=sec)

vname='5min'
sec=300
data2 = compute_breathing_instability_index(data2, vname=vname, sec=sec)


In [60]:
data.columns

Index(['IBI', 'IBI_mean_5min', 'IBI_std_5min', 'TVpB', 'TVpB_regularity',
       'TVpB_regularity_10sec', 'abd', 'acc_ai_10sec', 'acc_ai_1sec',
       'acc_enmo', 'acc_enmo_10sec', 'acc_enmo_1sec', 'accx', 'accx_1sec',
       'accy', 'accy_1sec', 'accz', 'accz_1sec', 'annotation', 'apnea', 'band',
       'cflow', 'chest', 'crcstatus', 'deriv1', 'epoch', 'exht', 'hypo_10_60',
       'inht', 'inht_cycle_ratio', 'inht_cycle_ratio_10sec', 'inht_exht_ratio',
       'inht_exht_ratio_10sec', 'katz_fd_10s_smoothed', 'katz_fd_30s_smoothed',
       'katz_fd_45s_smoothed', 'katz_fd_60s_smoothed', 'movavg_0_5s',
       'movavg_1_2s', 'movavg_1min', 'movmedian_10min', 'movmedian_1min',
       'movmedian_30sec', 'movmedian_5min', 'movstd_10min', 'movstd_10s',
       'movstd_12s', 'movstd_15s', 'movstd_1min', 'movstd_30s', 'movstd_30sec',
       'movstd_5min', 'movstd_60s', 'movstd_8s', 'peaks', 'position_cluster',
       'pr', 'ptaf', 'pulsequality', 'rr_10s', 'rr_10s_smooth', 'rr_60s',
       'samp

In [57]:
# plt.figure(figsize=(8,5))
# ax1 = plt.subplot(411)
# plt.plot(data['movavg_0_5s'])
# plt.subplot(412, sharex=ax1)
# plt.plot(data[f'ventilation0'])
# plt.subplot(413, sharex=ax1)
# # plt.plot(data[f'IBI_cvar_{vname}'])
# plt.plot(data[f'ventilation_10s'])
# plt.subplot(414, sharex=ax1)
# plt.plot(data[f'ventilation_10s_smooth'])

# # plt.subplot(414, sharex=ax1)
# # plt.plot(data['IBI'])

# # plt.subplot(414, sharex=ax1)
# # plt.plot(data[f'instability_index_{vname}'])

In [56]:
plt.figure(figsize=(8,5))
ax1 = plt.subplot(411)
plt.plot(data['movavg_0_5s'])
plt.subplot(412, sharex=ax1)
plt.plot(data[f'ventilation_cvar_{vname}'])
plt.ylim([0,1])
plt.subplot(413, sharex=ax1)
plt.plot(data[f'IBI_cvar_{vname}'])
plt.ylim([0,1])

# plt.plot(data[f'ventilation_{vname}'])
# plt.plot(data[f'ventilation0'])

# plt.subplot(414, sharex=ax1)
# plt.plot(data['IBI'])

plt.subplot(414, sharex=ax1)
plt.plot(data[f'instability_index_{vname}'])
plt.ylim([0,1])


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

(0, 1)

In [67]:
# vname = '30sec'
# sec = 30
# data[f'ventilation_{vname}'] = data['ventilation0'].rolling(vname).sum()/2
# data[f'ventilation_cvar_{vname}'] = data[f'ventilation_{vname}'].reset_index()[f'ventilation_{vname}'].rolling(sec*fs, center=True, min_periods=1).apply(lambda x: variation(x), raw=False).values


# vname ='1min'
# sec = 60
# data[f'ventilation_{vname}'] = data['ventilation0'].rolling(vname).sum()/2
# data[f'ventilation_cvar_{vname}'] = data[f'ventilation_{vname}'].reset_index()[f'ventilation_{vname}'].rolling(sec*fs, center=True, min_periods=1).apply(lambda x: variation(x), raw=False).values

# vname = '2min'
# sec = 120
# data[f'ventilation_{vname}'] = data['ventilation0'].rolling(vname).sum()/2
# data[f'ventilation_cvar_{vname}'] = data[f'ventilation_{vname}'].reset_index()[f'ventilation_{vname}'].rolling(sec*fs, center=True, min_periods=1).apply(lambda x: variation(x), raw=False).values
# data[f'IBI_cvar_{vname}'] = data['IBI'][data.peaks==1].reset_index()[f'ventilation_{vname}'].rolling(sec*fs, center=True, min_periods=1).apply(lambda x: variation(x), raw=False).values



# vname = '5min'
# sec = 300
# data[f'ventilation_{vname}'] = data['ventilation0'].rolling(vname).sum()/2
# data[f'ventilation_cvar_{vname}'] = data[f'ventilation_{vname}'].reset_index()[f'ventilation_{vname}'].rolling(sec*fs, center=True, min_periods=1).apply(lambda x: variation(x), raw=False).values



# data.loc[data.peaks==1, 'IBI_cvar_2min'] = data['IBI'][data.peaks==1].rolling('2min', min_periods=1).apply(lambda x: variation(x), raw=False)
# data['IBI_cvar_2min'].fillna(method='pad', inplace=True, limit=120*fs)


In [10]:
data['IBI_cvar_2min'] = np.nan
data.loc[data.peaks==1, 'IBI_cvar_2min'] = data['IBI'][data.peaks==1].rolling('2min', min_periods=1).apply(lambda x: variation(x), raw=False)
data['IBI_cvar_2min'].fillna(method='pad', inplace=True, limit=120*fs)

data2['IBI_cvar_2min'] = np.nan
data2.loc[data2.peaks==1, 'IBI_cvar_2min'] = data2['IBI'][data2.peaks==1].rolling('2min', min_periods=1).apply(lambda x: variation(x), raw=False)
data2['IBI_cvar_2min'].fillna(method='pad', inplace=True, limit=120*fs)

data['IBI_std_2min'] = np.nan
data.loc[data.peaks==1, 'IBI_std_2min'] = data['IBI'][data.peaks==1].rolling('2min', min_periods=1).std()
data['IBI_std_2min'].fillna(method='pad', inplace=True, limit=120*fs)

data2['IBI_std_2min'] = np.nan
data2.loc[data2.peaks==1, 'IBI_std_2min'] = data2['IBI'][data2.peaks==1].rolling('2min', min_periods=1).std()
data2['IBI_std_2min'].fillna(method='pad', inplace=True, limit=120*fs)


In [13]:
plt.figure(figsize=(8,5))
ax1 = plt.subplot(311)
plt.plot(data['ventilation_2min'])
plt.subplot(312, sharex=ax1)
plt.plot(data['ventilation_std_2min'])
plt.subplot(313, sharex=ax1)
plt.plot(data['ventilation_cvar_2min'])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/pandas/plotting/_matplotlib/converter.py:103: FutureWarning:

Using an implicitly registered datetime converter for a matplotlib plotting method. The converter was registered by pandas on import. Future versions of pandas will require you to explicitly register matplotlib converters.

To register the converters:
	>>> from pandas.plotting import register_matplotlib_converters
	>>> register_matplotlib_converters()



In [14]:
plt.figure(figsize=(8,5))
ax1 = plt.subplot(311)
plt.plot(data2['ventilation_2min'])
plt.subplot(312, sharex=ax1)
plt.plot(data2['ventilation_std_2min'])
plt.subplot(313, sharex=ax1)
plt.plot(data2['ventilation_cvar_2min'])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [15]:
plt.figure(figsize=(8,5))
ax1 = plt.subplot(311)
plt.plot(data['IBI'])
plt.subplot(312, sharex=ax1)
plt.plot(data['IBI_std_2min'])
plt.subplot(313, sharex=ax1)
plt.plot(data['IBI_cvar_2min'])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [16]:
plt.figure(figsize=(8,5))
ax1 = plt.subplot(311)
plt.plot(data2['IBI'])
plt.subplot(312, sharex=ax1)
plt.plot(data2['IBI_std_2min'])
plt.subplot(313, sharex=ax1)
plt.plot(data2['IBI_cvar_2min'])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [17]:
data['instability_index'] = (data['ventilation_cvar_2min'].values + data['IBI_cvar_2min'].values)/2
data2['instability_index'] = (data2['ventilation_cvar_2min'].values + data2['IBI_cvar_2min'].values)/2

In [77]:
%load_ext autoreload

%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [78]:
signals_to_plot

['acc_ai_10sec',
 'movavg_0_5s',
 'movavg_1min',
 'rr_10s_smooth',
 'IBI',
 'IBI_mean_5min',
 'IBI_cvar_30sec',
 'ventilation_cvar_2min',
 'ventilation_10s_smooth',
 'ventilation_2min',
 'instability_index_30sec',
 'instability_index_1min',
 'instability_index_2min',
 'instability_index_5min',
 'inht_cycle_ratio_10sec']

In [84]:
fig1 = icu_sleep_plot(data[signals_to_plot])
plot(fig1, filename = 'tmp_breathing_stability.html')

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/pandas/core/frame.py:4223: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



'tmp_breathing_stability.html'

In [85]:
fig1 = icu_sleep_plot(data2[signals_to_plot])
plot(fig1, filename = 'tmp_breathing_stability2.html')

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/pandas/core/frame.py:4223: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



'tmp_breathing_stability2.html'